# Chapter 9: LLMOps and Deployment

Estimated time: ~6 hours.

Prerequisites: Chapter 4 (reliability patterns), Chapter 5 (cost/latency), Chapter 8 (this
chapter is where "how will you know it's working, in production, over time," question 8 of
Chapter 8's framework, actually gets answered).

## Setup

This chapter makes no model calls. Responses come from a deterministic simulator so
canary comparisons are reproducible run to run.

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random

from agentlib.grading import check
from agentlib import llm_client

random.seed(42)
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print("This chapter makes no model calls: responses come from a deterministic simulator so\n"
      "a canary comparison is reproducible run to run, which a real model would not be.\n"
      "HAS_KEY is printed for consistency with the other chapters, not because anything\n"
      "here branches on it.")

LLM_PROVIDER = 'anthropic', HAS_KEY = False
This chapter makes no model calls: responses come from a deterministic simulator so
a canary comparison is reproducible run to run, which a real model would not be.
HAS_KEY is printed for consistency with the other chapters, not because anything
here branches on it.


## Section 1: Definitions

### Prompt versioning

A prompt edited in place, with no record of what changed or when, is this chapter's version
of Chapter 4's stale-cache lesson: the failure is invisible until something goes wrong, and
by then there is no record of what the "before" state even was. Treat every prompt change as
a new, immutable version, never an edit, with a pointer (`current_version`) that gets moved,
not a value that gets overwritten. Rollback then means moving the pointer, not trying to
reconstruct what the prompt used to say.

Real-world parallels: Terraform state files are versioned and immutable. Git commits are
immutable by design (amending creates a new commit, it does not edit the old one). Database
migration systems (Alembic, Flyway) never edit a migration that already ran.

### Canary releases and progressive delivery

Route a small percentage of real traffic to a new version, watch it, and ramp up only if it
looks healthy. The same pattern web services use for code deploys, applied to a prompt or
model swap. The stakes are different, though. A bad code deploy usually fails loudly (a 500
error, a crash); a bad prompt change often fails quietly: slightly worse answers, a subtly
higher refusal rate, nothing that trips an infrastructure alert.

Real-world parallels: AWS CodeDeploy canary deployments, Kubernetes rolling updates with
health checks, LaunchDarkly feature flags with percentage rollouts.

### Shadow deployment

Run the new version on real traffic without serving its output to users. Log what it would
have said, compare against the current version offline, and only promote to a real canary
once the shadow comparison looks good. Strictly safer than a canary for high-stakes changes,
at the cost of not measuring real user reactions until the canary stage starts.

Real-world parallels: Netflix runs shadow traffic against new recommendation models before
promoting them. Search engines evaluate new ranking algorithms on logged queries before
any user sees the results.

### Drift detection

Two distinct kinds.

**Behavioral drift** happens in your own system: the new prompt or model version's output
distribution has shifted. This chapter's build section detects this directly.

**Upstream drift** is different: the underlying provider silently updates a model version
behind a stable-sounding API name, and your system's behavior shifts without you having
changed anything at all. A recurring failure mode with hosted LLM APIs that has no
classical-software equivalent.

Real-world parallels: Evidently AI and WhyLabs provide drift monitoring dashboards.
OpenAI's `gpt-4` endpoint has historically pointed at different model snapshots over time
without changing the model name.

### Containerization

Packaging an application with its dependencies into a reproducible, isolated unit.
For LLM applications, the key concerns are: layer caching (install dependencies before
copying source code), least-privilege (non-root user), no baked secrets (API keys at
runtime, never at build time), and health checks (knowing the container is running is not
the same as knowing it is healthy).

Real-world parallels: every major cloud provider's managed container service (ECS, Cloud
Run, AKS). This repository's own root-level `Dockerfile` demonstrates all four concerns.

## Section 2: Concept Explanation

### Immutable version chain

```
publish(v1, text_a)    publish(v2, text_b)    publish(v3, text_c)
       |                      |                      |
       v                      v                      v
   +--------+            +--------+            +--------+
   | v1     |            | v2     |            | v3     |
   | text_a |            | text_b |            | text_c |
   +--------+            +--------+            +--------+

   current_version pointer:

   promote(v1)  -->  [v1]         (history: [v1])
   promote(v2)  -->  [v2]         (history: [v1, v2])
   promote(v3)  -->  [v3]         (history: [v1, v2, v3])
   promote(v1)  -->  [v1]         (history: [v1, v2, v3, v1])  <-- rollback
```

Rollback is a pointer move. The text at v1 never changed. The history records every
promote, including rollbacks, so an incident review sees exactly when each version was
live.

### Canary routing via sticky hashing

```
  request_id
      |
      v
  sha256(request_id)
      |
      v
  hash_value % 100
      |
      +-- < canary_pct  -->  canary version
      |
      +-- >= canary_pct -->  stable version
```

Two properties fall out of hashing instead of using `random()`:
- Same request id always routes the same way, in any process, on any day.
- Raising `canary_pct` only adds requests to the canary cohort, never moves any out.

### Shadow deployment sequence

```
  real traffic
      |
      +---> stable version ---> serve to user
      |
      +---> shadow version ---> log only (never served)
                |
                v
         offline comparison
                |
                v
         if acceptable --> promote to canary
         if not        --> iterate on prompt
```

### Trade-offs

| Decision | Low setting | High setting |
|---|---|---|
| Canary percentage | 1%: safe but slow to get signal | 50%: fast signal but half your users hit a potential regression |
| Health threshold | 0.01: catches small shifts but fires on noise | 0.10: misses moderate regressions |
| Sample size | 50: fast checks, low statistical power | 500: slower checks, reliable signal |

## Section 3: Example Code Segments

### Deterministic response simulator

Stands in for real model calls. Generates a synthetic but deterministic outcome per
(prompt text, request), so canary and drift demos produce the same numbers every run.

In [ ]:
import hashlib


def simulate_response(prompt_text: str, request_id: str) -> dict:
    '''Deterministic stand-in for a real model call. Two independent signals derived from a
    hash of (prompt text, request id): a refusal probability and a response-length range.
    Real behavior differences between prompt versions are simulated by prompt TEXT actually
    affecting these signals below -- not hardcoded per version_id -- so this genuinely reacts
    to what a prompt says, the same way a real model's behavior would.'''
    h = int(hashlib.sha256(f"{prompt_text}::{request_id}".encode()).hexdigest(), 16)

    base_refusal_rate = 0.03
    if "cautious" in prompt_text.lower() or "when in doubt, decline" in prompt_text.lower():
        base_refusal_rate = 0.35  # a prompt rewrite that overcorrects toward refusing

    refused = (h % 1000) / 1000 < base_refusal_rate
    length = 80 + (h % 150)
    return {"refused": refused, "length": length}


sample = simulate_response("You are a helpful support assistant.", "req-00001")
print(f"Sample response signal: {sample}")


### Shadow comparison

Runs both versions on the same requests, serves only the stable response, and compares
offline. The `served_to_users` field makes the contract explicit: shadow output is never
shown to anyone.

In [ ]:
def shadow_compare(request_ids: list, stable_text: str, shadow_text: str) -> dict:
    stable_results = [simulate_response(stable_text, rid) for rid in request_ids]
    shadow_results = [simulate_response(shadow_text, rid) for rid in request_ids]

    stable_refusal_rate = sum(r["refused"] for r in stable_results) / len(stable_results)
    shadow_refusal_rate = sum(r["refused"] for r in shadow_results) / len(shadow_results)

    return {
        "stable_refusal_rate": stable_refusal_rate,
        "shadow_refusal_rate": shadow_refusal_rate,
        "delta": shadow_refusal_rate - stable_refusal_rate,
        "served_to_users": "stable only -- shadow output never shown, this comparison is offline",
    }


demo_ids = [f"req-{i:05d}" for i in range(100)]
comparison = shadow_compare(
    demo_ids,
    "You are a helpful support assistant.",
    "You are a cautious support assistant. When in doubt, decline to answer.",
)
for k, v in comparison.items():
    print(f"{k}: {v}")


### Progressive rollout harness

An automated ramp-up: start at a small canary percentage, check health after each stage,
ramp up if healthy, halt and roll back if not. Used in both the Break It section (with
the buggy threshold) and after the fix.

In [ ]:
def run_progressive_rollout(stable_text: str, canary_text: str, health_check, stages=(5, 25, 50, 100)) -> dict:
    """Automated canary ramp-up. Runs health_check at each stage percentage."""
    log = []
    for pct in stages:
        sample_ids = [f"rollout-req-{i:05d}" for i in range(500)]
        stable_results = [simulate_response(stable_text, rid) for rid in sample_ids]
        canary_results = [simulate_response(canary_text, rid) for rid in sample_ids]
        stable_rate = sum(r["refused"] for r in stable_results) / len(stable_results)
        canary_rate = sum(r["refused"] for r in canary_results) / len(canary_results)

        healthy = health_check(stable_rate, canary_rate)
        log.append({"stage_pct": pct, "stable_refusal_rate": stable_rate,
                     "canary_refusal_rate": canary_rate, "healthy": healthy})
        if not healthy:
            return {"outcome": "ROLLED BACK", "final_stage_pct": pct, "log": log}
    return {"outcome": "FULLY PROMOTED", "final_stage_pct": 100, "log": log}


### The Dockerfile

This repository has a real, working root-level `Dockerfile`. Key choices worth calling
out:

- `requirements.txt` copied and installed before source code, so editing a notebook does
  not invalidate the dependency-install layer in Docker's build cache.
- Container runs as `agent`, not `root` (least privilege at the container boundary).
- `LLM_PROVIDER` and API keys supplied at `docker run` time, never baked into a layer
  (a secret in a Docker layer is recoverable from the image itself).
- `HEALTHCHECK` included: knowing a container is running is not the same as knowing it
  is healthy.

In [15]:
dockerfile_path = _repo_root / "Dockerfile"
dockerfile_text = dockerfile_path.read_text()
print(dockerfile_text)


# Containerizes this repository's Python environment so the curriculum's agent code (the
# capstone agent, or any individual chapter's notebook run headlessly) can run the same way
# regardless of the host machine. See curriculum/09_llmops_deployment.ipynb for the
# containerization concept section this file accompanies, and PROGRESS.md's Unit 10 notes
# for how this file was verified in this repo's own build environment.

FROM python:3.11-slim

# A non-root user to run the agent as -- least privilege inside the container too, not just
# in the agent's own tool scoping (Chapter 6's principle, applied one layer down).
RUN useradd --create-home --uid 1000 agent
WORKDIR /app

# Copy dependency manifests first so Docker's layer cache is only invalidated by a real
# dependency change, not by every source-code edit -- the single most common Dockerfile
# performance mistake, worth doing correctly even in a course example.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.

### A note on verifying this in the environment this course was built in

This repo's own build environment has a working Docker daemon and CLI, but its sandboxed
network policy blocks the CDN hosts Docker Hub and GitHub Container Registry actually serve
image layer blobs from (`production.cloudfront.docker.com`,
`pkg-containers.githubusercontent.com`). This was confirmed by directly attempting `docker
pull python:3.11-slim` and a GHCR image, both of which fail at the blob-download step
specifically, not at the registry-API step. A `FROM scratch` build with no external base
image completes instantly in the same environment, confirming this is genuinely a
network-policy block on those two CDNs, not a broader problem with Docker itself. This means
`docker build` on the Dockerfile above could not be executed end-to-end and verified in this
specific build session: a real, honestly-reported limitation, not something worked around
with a substitute (there's no alternate "real" source for a base container image the way
there was for SQuAD or `tiktoken`'s data in earlier chapters). Anyone running this in a
normal network environment should expect `docker build -t agent-interview-prep .` to work
correctly; see `PROGRESS.md`'s Unit 10 notes for the full verification trail.

## Section 4: Build It Yourself

Three graded tasks: an immutable prompt registry, a sticky canary router, and a
calibrated drift-detection threshold.

### Task 1: `PromptRegistry` (immutable prompt versioning)

`PromptRegistry` enforces immutability in code, not as a convention someone has to
remember: `publish()` refuses to overwrite an existing version ID, and `promote()` is
the only way to change what is live. Rollback is then just "promote an older version,"
not a special operation.

The registry is yours to build, and "immutable" is the load-bearing word. A store that
lets you edit a published version passes every test that writes a prompt and reads it
straight back. It fails exactly once, in production, on the day you try to roll back.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PromptVersion:
    '''frozen=True is not decoration. It is what stops a published prompt being edited in
    place, which is what makes the audit trail describe prompts rather than just ids.'''
    version_id: str
    text: str


class PromptRegistry:
    '''An append-only store of prompt versions, plus a pointer at whichever one is live.

    - publish(version_id, text) -> PromptVersion. Store it and return it. Raise ValueError
      if that id already exists: versions are immutable, so a change means a new id.
    - promote(version_id) -> None. Point current_version at it and append it to history.
      Raise ValueError if the id was never published.
    - get(version_id) -> PromptVersion.

    Publishing is not promoting. A new version goes on the shelf; promoting is what points
    traffic at it, and conflating the two makes every publish a deploy.

    history is the audit trail, so append on EVERY promote -- including a promote back to a
    version that ran before. That repeat is the rollback, and it is the entry an incident
    review goes looking for.
    '''

    def __init__(self):
        self._versions: dict[str, PromptVersion] = {}
        self.current_version: str | None = None
        self.history: list[str] = []

    def publish(self, version_id: str, text: str) -> PromptVersion:
        raise NotImplementedError("Implement me, then re-run this cell")

    def promote(self, version_id: str) -> None:
        raise NotImplementedError("Implement me, then re-run this cell")

    def get(self, version_id: str) -> PromptVersion:
        raise NotImplementedError("Implement me, then re-run this cell")


PromptRegistry = check("ch09-prompt-version", PromptRegistry)

In [3]:
registry = PromptRegistry()
registry.publish("v1", "You are a helpful support assistant. Answer the customer's question directly.")
registry.promote("v1")
print(f"current_version = {registry.current_version!r}")
print(f"history = {registry.history}")

current_version = 'v1'
history = ['v1']


### Task 2: `route_request` (sticky canary routing)

A sticky router: the same `request_id` always routes to the same version, so a given
user gets a consistent experience across a session instead of flip-flopping between
prompt versions call to call.

Hash the id and split on the hash. That buys you the ramp property for free: widening
the canary from 5% to 25% adds people to the cohort without moving anyone out of it.

In [ ]:
def route_request(request_id: str, stable_version: str, canary_version: str, canary_pct: float) -> str:
    '''Decide which prompt version this request sees.

    Return canary_version for canary_pct percent of requests and stable_version for the rest.
    The assignment must be a pure function of request_id -- hash it (hashlib.sha256 on the
    encoded id), take the hash modulo 100, and compare against canary_pct.

    Two properties fall out of doing it that way, and both matter:
      - the same request id always routes the same way, in any process, on any day
      - raising canary_pct only ever ADDS requests to the canary cohort, never moves any out

    canary_pct=0 means nobody, canary_pct=100 means everybody.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


route_request = check("ch09-canary-split", route_request)

In [6]:
registry.publish("v2", "You are a helpful support assistant. Answer the customer's question directly and concisely.")

request_ids = [f"req-{i:05d}" for i in range(1000)]
routed = [route_request(rid, "v1", "v2", canary_pct=10) for rid in request_ids]
canary_share = routed.count("v2") / len(routed) * 100
print(f"Canary share of traffic: {canary_share:.1f}% (target: 10%)")

Canary share of traffic: 9.3% (target: 10%)


### Task 3: `check_canary_health_fixed` (calibrated drift detection)

The threshold is the entire content of this function, and it is wrong in both directions
if you are careless with it. Too loose and it never fires. Too tight and it fires on every
rollout, because two 500-request samples never produce identical rates. A rollback signal
that cries wolf gets muted inside a week.

A difference of exactly the threshold is not inside it. The default of 0.05 is calibrated
against what a real regression looks like: five percentage points, not fifty.

In [ ]:
def check_canary_health_fixed(stable_rate: float, canary_rate: float, threshold: float = 0.05) -> bool:
    '''Is the canary's behaviour close enough to the stable version's to keep rolling?

    Return True when the two rates are within `threshold` of each other, False otherwise.
    Compare the absolute difference: a canary that refuses thirty points LESS often has also
    changed sharply, and an unexplained improvement is worth stopping a rollout for too.

    A difference of exactly the threshold is not inside it. The default of 0.05 is calibrated
    against what a real regression looks like -- five percentage points, not fifty -- so it
    has to catch this chapter's 32-point swing without anyone passing an explicit threshold.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


check_canary_health_fixed = check("ch09-drift-detect", check_canary_health_fixed)

In [ ]:
registry.publish("v3-regressed",
    "You are a cautious support assistant. When in doubt, decline to answer.")
registry.promote("v3-regressed")

result_fixed = run_progressive_rollout(registry.get("v1").text, registry.get("v3-regressed").text, check_canary_health_fixed)
print(f"Outcome: {result_fixed['outcome']}")
for entry in result_fixed["log"]:
    print(f"  {entry['stage_pct']:3d}% -- stable={entry['stable_refusal_rate']:.3f} "
          f"canary={entry['canary_refusal_rate']:.3f} healthy={entry['healthy']}")


## Section 5: Playground

Experiments with the deployment machinery you just built. Change one parameter at a
time and observe how the system's behavior shifts.

### Experiment 1: Canary percentage

Try `canary_pct` values of 1, 5, 10, 25, and 50. For each, measure the actual share of
traffic routed to the canary. At what percentage do you get enough signal to detect a
regression quickly?

In [ ]:
# --- TRY DIFFERENT VALUES ---
for pct in [1, 5, 10, 25, 50]:
    routed = [route_request(f"req-{i:05d}", "v1", "v2", canary_pct=pct)
              for i in range(1000)]
    actual = routed.count("v2") / len(routed) * 100
    print(f"  canary_pct={pct:3d}  actual={actual:.1f}%")


### Experiment 2: Health check threshold

Run `check_canary_health_fixed` with thresholds of 0.01, 0.05, 0.10, and 0.50 against
the regressed prompt. Which thresholds catch the regression? Which would also fire on
normal run-to-run variation?

In [ ]:
# --- TRY DIFFERENT THRESHOLDS ---
stable_text = registry.get("v1").text
canary_text = registry.get("v3-regressed").text
sample_ids = [f"threshold-req-{i:05d}" for i in range(500)]
stable_rate = sum(simulate_response(stable_text, r)["refused"] for r in sample_ids) / len(sample_ids)
canary_rate = sum(simulate_response(canary_text, r)["refused"] for r in sample_ids) / len(sample_ids)
print(f"Stable refusal rate: {stable_rate:.3f}")
print(f"Canary refusal rate: {canary_rate:.3f}")
print(f"Delta: {abs(canary_rate - stable_rate):.3f}")
print()
for thresh in [0.01, 0.05, 0.10, 0.50]:
    healthy = check_canary_health_fixed(stable_rate, canary_rate, threshold=thresh)
    print(f"  threshold={thresh:.2f}  healthy={healthy}  (catches regression: {not healthy})")


### Experiment 3: Sample size and statistical power

Run the shadow comparison with sample sizes of 10, 50, 100, and 500. How stable is the
measured delta across sizes? At what sample size does the signal become reliable?

In [ ]:
# --- TRY DIFFERENT SAMPLE SIZES ---
stable_text = registry.get("v1").text
canary_text = registry.get("v3-regressed").text
for n in [10, 50, 100, 500]:
    ids = [f"sample-req-{i:05d}" for i in range(n)]
    result = shadow_compare(ids, stable_text, canary_text)
    print(f"  n={n:4d}  stable={result['stable_refusal_rate']:.3f}  "
          f"shadow={result['shadow_refusal_rate']:.3f}  delta={result['delta']:.3f}")


### Experiment 4: Hash stickiness

Verify that the same request id always routes to the same version, regardless of when
or how many times you call `route_request`. Also verify that raising `canary_pct` only
adds requests to the canary cohort, never removes any.

In [ ]:
# --- VERIFY STICKINESS ---
test_ids = [f"sticky-{i}" for i in range(100)]
run1 = [route_request(rid, "v1", "v2", canary_pct=10) for rid in test_ids]
run2 = [route_request(rid, "v1", "v2", canary_pct=10) for rid in test_ids]
print(f"Same routing across two runs: {run1 == run2}")

# Verify ramp-up only adds, never removes
at_10 = set(rid for rid, v in zip(test_ids, run1) if v == "v2")
at_25 = set(rid for rid, v in zip(test_ids,
    [route_request(rid, "v1", "v2", canary_pct=25) for rid in test_ids]) if v == "v2")
print(f"10% canary ids are subset of 25% canary ids: {at_10.issubset(at_25)}")
print(f"10% count: {len(at_10)}, 25% count: {len(at_25)}")


## Section 6: Break It

### Break It 1: an in-place prompt edit that makes "rollback" a no-op

This is a plain mutable store: exactly what "just quickly tweak the prompt in the config"
looks like in code, with no registry enforcing anything.

In [8]:
class NaivePromptStore:
    '''The bug: prompt text lives in a plain mutable dict, edited in place. No history, no
    immutability -- nothing stops the same key from silently pointing at different text
    over time.'''
    def __init__(self):
        self.prompts: dict[str, str] = {}

    def set(self, version_id: str, text: str) -> None:
        self.prompts[version_id] = text  # overwrites silently if version_id already exists


naive = NaivePromptStore()
naive.set("v1", "You are a helpful support assistant. Answer the customer's question directly.")
print("Before edit:", naive.prompts["v1"])

# Someone "quickly tweaks" the live prompt to be more conservative -- still under the same
# key, v1, because nobody thought of this as "publishing a new version."
naive.set("v1", "You are a cautious support assistant. When in doubt, decline to answer.")
print("After  edit:", naive.prompts["v1"])

print("\n'Rolling back to v1' does nothing -- v1 IS the edited, regressed text now:")
print(f"  {naive.prompts['v1']!r}")


Before edit: You are a helpful support assistant. Answer the customer's question directly.
After  edit: You are a cautious support assistant. When in doubt, decline to answer.

'Rolling back to v1' does nothing -- v1 IS the edited, regressed text now:
  'You are a cautious support assistant. When in doubt, decline to answer.'


Diagnose before reading on. What would you check to confirm this is what happened, and
what does the fix actually need to guarantee?

**Hint 1**: What happens when you overwrite v1 with v2? Can you get the original v1 text
back?

**Hint 2**: Compare `NaivePromptStore.set()` (allows overwrite) with
`PromptRegistry.publish()` (raises on duplicate). Which one makes rollback possible?

**Production impact**: A team deploys a regressed prompt, notices quality dropped, and
tries to roll back. The rollback command succeeds (no error), but quality does not
recover, because the text the pointer points at was silently overwritten weeks ago.
The incident review finds no record of what the original prompt said.

**Interview follow-up**: "Your team deployed a new prompt and quality dropped. How do
you roll back, and what if the old version was overwritten?"

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_ROLLBACK_DIAGNOSIS = """Replace this with your diagnosis."""


MY_ROLLBACK_DIAGNOSIS = check("ch09-diagnose-mutable-prompt", MY_ROLLBACK_DIAGNOSIS)

In [ ]:
# v3-regressed was published above in Build It; just promote it to make it live again
registry.promote("v3-regressed")
print(f"Live now: {registry.current_version} -> {registry.get(registry.current_version).text!r}")

# The regression is noticed -- roll back.
registry.promote("v1")
print(f"\nRolled back to: {registry.current_version}")
print(f"v1's text, completely unchanged by anything that happened to v3-regressed:")
print(f"  {registry.get('v1').text!r}")

print("\nAnd the mistake from the naive store above is now structurally impossible:")
try:
    registry.publish("v1", "some completely different text")
except ValueError as e:
    print(f"  Correctly rejected: {e}")


### Break It 2: a canary rollout that ships a real regression

An automated progressive rollout with a drift-detection threshold set so loosely that it
never catches anything. A 50-percentage-point threshold sounds "safe and conservative"
if you are not thinking about what a real regression's magnitude actually looks like.
This chapter's real regression is "only" about 32 points, comfortably under it.

In [ ]:
def check_canary_health(stable_rate: float, canary_rate: float, threshold: float = 0.5) -> bool:
    """The bug: threshold=0.5 means a 50-percentage-point swing is needed to trigger."""
    return abs(canary_rate - stable_rate) < threshold


result = run_progressive_rollout(
    registry.get("v1").text,
    registry.get("v3-regressed").text,
    check_canary_health
)
print(f"Outcome: {result['outcome']}")
for entry in result["log"]:
    print(f"  {entry['stage_pct']:3d}% -- stable={entry['stable_refusal_rate']:.3f} "
          f"canary={entry['canary_refusal_rate']:.3f} healthy={entry['healthy']}")


The regressed prompt just shipped to 100% of traffic. What is wrong with
`check_canary_health`, specifically?

**Hint 1**: Look at the delta between stable and canary refusal rates. Is 0.33 less
than 0.5?

**Hint 2**: What threshold would actually catch a 33-percentage-point swing? What
threshold would also fire on normal run-to-run noise (1-2 percentage points)?

**Production impact**: A prompt rewrite that triples the refusal rate ships to 100% of
users. No automated check fires. The regression is discovered a week later by a
stakeholder noticing more customer complaints.

**Interview follow-up**: "You are picking a drift-detection threshold for a new
deployment with no historical data. How do you arrive at a number rather than picking
a round-sounding one?"

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_THRESHOLD_DIAGNOSIS = """Replace this with your diagnosis."""


MY_THRESHOLD_DIAGNOSIS = check("ch09-diagnose-loose-threshold", MY_THRESHOLD_DIAGNOSIS)

## Section 7: Interview Q&A

### Question 1: "How do you safely deploy a new prompt version to production?"

Treat every prompt change as a new immutable version in a registry. Deploy it first as a
shadow (run on real traffic, log output, serve nothing) to catch obvious regressions
offline. Then promote to a small canary (5-10% of traffic) with sticky hash-based routing
so the same user stays on the same version. Monitor canary health with a calibrated
threshold (not a round number, but one derived from your system's normal run-to-run
variation). Ramp up only when the canary looks healthy. If it does not, rollback is a
pointer move because the old version's text was never overwritten.

### Question 2: "What is the difference between a canary release and an A/B test?"

A canary release is a safety mechanism: route a small fraction of traffic to the new
version, watch for regressions, and roll back fast if something breaks. The goal is safe
deployment, not measurement. An A/B test is a measurement mechanism: split traffic
50/50 (or some planned ratio) between two versions and run long enough to reach
statistical significance on a specific metric. A canary starts small and ramps up; an
A/B test holds its split constant. You can use a canary as the first stage of an A/B
test, but they serve different purposes.

### Question 3: "How do you detect that a model update degraded your agent's performance?"

Two monitoring layers. First, track your own behavioral metrics (refusal rate, response
length distribution, task success rate) continuously against a baseline. A statistically
significant shift in any of these, without a corresponding prompt or code change on your
side, signals upstream drift: the provider changed the model behind a stable API name.
Second, run a fixed evaluation set (Chapter 3's offline metrics) on a schedule. If
scores drop without your system changing, the model changed under you. The fix is to pin
model versions when the provider offers them, and to treat an unpinned model name as a
deployment risk, not a convenience.

### Question 4: "Walk me through your Dockerfile for an LLM application."

Four things a reviewer checks. (1) Layer order: copy `requirements.txt` and install
dependencies before copying source code, so a code edit does not invalidate the
dependency-install cache layer. (2) Non-root user: the container runs as a service
account, not root, applying least privilege at the container boundary. (3) No baked
secrets: API keys and `LLM_PROVIDER` are supplied at `docker run` time via environment
variables, never copied into a layer at build time. A secret in a Docker layer is
recoverable from the image itself even if a later layer removes it. (4) `HEALTHCHECK`:
a cheap probe that confirms the application inside the container is functional, not just
that the process is alive.

### System design drill

Answer each of these on your own before checking the model answers.

In [16]:
from agentlib.self_check import drill as open_drill

drill = open_drill(9)
drill.questions()

Chapter 9 written drill — 4 questions

1. Cold diagnosis: refusals climbed, no infrastructure alert fired
2. Design judgment: skip shadow deployment, go straight to a 5% canary
3. Judgment call: picking a threshold with no historical data
4. Conceptual: silent upstream model drift


#### Answering these

There is a slot below for each question. Write your answer into it, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side.

`check(n)` will not show you an answer until you have written one of your own -- once you have
read the model answer you can no longer find out what you actually knew. If you want it
anyway, `drill.reveal(n)` is there and makes no judgement.

Answers are read from `solutions/ch09_*_answers.md` at runtime, so nothing in this
notebook contains one.

In [17]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 9: 0/4 answered
  still open: [1, 2, 3, 4]


In [18]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  Cold diagnosis: refusals climbed, no infrastructure alert fired

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References

1. Nygard, M. (2007). *Release It!* Pragmatic Bookshelf. Circuit breakers, canary
   releases, and progressive delivery patterns.
2. AWS deployment patterns: CodeDeploy canary, blue/green, and linear deployment
   configurations.
3. Kubernetes documentation: rolling updates, readiness probes, and canary deployments.
4. Docker best practices: multi-stage builds, layer caching, non-root users, and
   `HEALTHCHECK` directives.
5. Evidently AI and WhyLabs: open-source and managed drift monitoring for ML systems.
6. Related chapters: Ch4 (reliability patterns underlying retries and circuit breakers),
   Ch5 (cost monitoring and model routing), Ch8 (system design framework, question 8).

## Next: Interview Prep and Capstone

With all nine chapters built, `interview_prep/` assembles the cross-chapter question
bank and mock-interview tooling this course has been pointing at since Chapter 1's
solutions-file convention, and the capstone puts everything from Chapters 1-9 into one
project.